In [ ]:
import sys
from pathlib import Path

# Add the src folder to the Python path so Jupyter can find your new module
sys.path.append(str(Path('../src').resolve()))

# Import the class you just built
from api_client import AESClient

# Initialize the client
client = AESClient()

# Execute a massive raw data pull with ONE line of code
client.get_all_team_data(185651)


In [ ]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append(str(Path('../src').resolve()))
from transformer import AESTransformer

# Initialize the transformer
transformer = AESTransformer()

# Process all raw files for our target team
transformer.process_all(185651)

# Let's inspect the final, clean output for matches!
clean_matches_path = Path('../Data/processed/team_185651_matches.csv')

if clean_matches_path.exists():
    df_clean_matches = pd.read_csv(clean_matches_path)
    print("\n--- FINAL ELT OUTPUT: CLEAN MATCH HISTORY ---")
    display(df_clean_matches.head(10))


In [ ]:
from pathlib import Path
import json

raw_dir = Path('../Data/raw/aes_api/team_123957')

print("--- RAW DATA DIRECTORY CHECK ---")
if not raw_dir.exists():
    print(f"[!] Directory not found: {raw_dir.absolute()}")
else:
    files = list(raw_dir.glob('*.json'))
    print(f"[SUCCESS] Found {len(files)} files in raw data lake:")
    
    for f in files:
        # Check file size
        size_kb = f.stat().st_size / 1024
        print(f" - {f.name} ({size_kb:.1f} KB)")
        
    # Check specifically for match files
    match_files = list(raw_dir.glob('matches_event_*.json'))
    print(f"\n[INFO] Match specific files found: {len(match_files)}")
    
    processed_matches = Path('../Data/processed/team_123957_matches.csv')
    print(f"[INFO] Did the transformer create the CSV? {processed_matches.exists()}")


In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Reload module
if 'transformer' in sys.modules:
    del sys.modules['transformer']

sys.path.append(str(Path('../src').resolve()))
from transformer import AESTransformer

# Run transformation
transformer = AESTransformer()
transformer.process_all(185651)

# Check the results
matches_csv = Path('../Data/processed/team_185651_matches.csv')
if matches_csv.exists():
    print("\n[SUCCESS] The ELT pipeline is fully operational! Here is the clean data:")
    display(pd.read_csv(matches_csv).head(10))
else:
    print("\n[FAILED] CSV still missing. Check the logs above.")


In [ ]:
import sys
import pandas as pd
import time
from pathlib import Path

sys.path.append(str(Path('../src').resolve()))
from api_client import AESClient

# Configuration
TARGET_X = 185651
DEGREES = 2
LOCAL_MATCH_DB = Path('../Data/processed/events/master_match_results.csv')
RAW_STORAGE = Path('../Data/raw/aes_api')

# Phase 1: Mapping
df = pd.read_csv(LOCAL_MATCH_DB).dropna(subset=['Team_A_ID', 'Team_B_ID'])
df['Team_A_ID'] = df['Team_A_ID'].astype(int)
df['Team_B_ID'] = df['Team_B_ID'].astype(int)

def find_opponents(ids):
    a = df[df['Team_A_ID'].isin(ids)]['Team_B_ID']
    b = df[df['Team_B_ID'].isin(ids)]['Team_A_ID']
    return set(a).union(set(b))

tiers = {0: {TARGET_X}}
visited = {TARGET_X}

for i in range(1, DEGREES + 1):
    opps = find_opponents(tiers[i-1]) - visited
    tiers[i] = opps
    visited.update(opps)
    print(f"Degree {i}: Found {len(opps)} new unique teams.")

# Phase 2: Extraction
queue = []
for i in range(DEGREES + 1): queue.extend(list(tiers[i]))

client = AESClient()
for i, t_id in enumerate(queue, 1):
    meta = RAW_STORAGE / f"team_{t_id}/metadata.json"
    if meta.exists(): continue
    
    print(f"[{i}/{len(queue)}] Extracting Team ID: {t_id}")
    client.get_all_team_data(t_id)
    time.sleep(1.2)
